In [5]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv("NEBIUS_API_KEY")
)


In [6]:
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-Coder-32B-Instruct",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": """You're a C code expert. You write high quality C code without being too verbose. The user will ask you to produce some C programs. You write the code only, do not add any other text or comment.

Here is an example of a correct answer that is expected from you as an AI assistant:

User: Generate a program that takes X and Y values from a CSV file called \"data.csv\", calculate the weight of a linear regression model and then print the weights of the model in a file called \"result\"

AI Assistant: #include <stdio.h>
#include <stdlib.h>

void linear_regression(const char *filename, double *w0, double *w1) {
    FILE *file = fopen(filename, \"r\");
    if (!file) return;
    
    double x, y, sum_x = 0, sum_y = 0, sum_xy = 0, sum_x2 = 0;
    int n = 0;
    
    while (fscanf(file, \"%lf,%lf\", &x, &y) == 2) {
        sum_x += x;
        sum_y += y;
        sum_xy += x * y;
        sum_x2 += x * x;
        n++;
    }
    fclose(file);
    
    if (n > 1) {
        double denom = n * sum_x2 - sum_x * sum_x;
        if (denom != 0) {
            *w1 = (n * sum_xy - sum_x * sum_y) / denom;
            *w0 = (sum_y - (*w1) * sum_x) / n;
        }
    }
}

int main() {
    double w0 = 0, w1 = 0;
    linear_regression(\"data.csv\", &w0, &w1);
    
    FILE *output = fopen(\"result\", \"w\");
    if (output) {
        fprintf(output, \"w0: %lf\nw1: %lf\n\", w0, w1);
        fclose(output);
    }
    
    return 0;
}"""
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": """Write an algorithm to calculate the similarity of two text strings."""
                }
            ]
        }
    ]
)

print(response.to_json())

{
  "id": "chatcmpl-b8ad0005d8244556ad35c74ee2547aa4",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "#include <stdio.h>\n#include <string.h>\n#include <ctype.h>\n#include <math.h>\n\n#define MAX_LEN 1000\n\nvoid to_lowercase(char *str) {\n    for (int i = 0; str[i]; i++) str[i] = tolower(str[i]);\n}\n\nint count_words(const char *str, char *word) {\n    int count = 0;\n    char *token = strtok(strdup(str), \" \");\n    while (token) {\n        if (strcmp(token, word) == 0) count++;\n        token = strtok(NULL, \" \");\n    }\n    return count;\n}\n\ndouble cosine_similarity(char *text1, char *text2) {\n    to_lowercase(text1);\n    to_lowercase(text2);\n    \n    char *words[MAX_LEN];\n    int freq1[MAX_LEN] = {0}, freq2[MAX_LEN] = {0}, unique_words = 0;\n    \n    char *token = strtok(text1, \" \");\n    while (token) {\n        int found = 0;\n        for (int i = 0; i < unique_words; i++) {\n    

In [13]:
def extract_functions(code):
    """Extracts compression and decompression functions from C code using comment delimiters"""
    compress_pattern = re.compile(r'//INIZIO_FUNZIONE_COMPRESSIONE\n(.*?)//FINE_FUNZIONE_COMPRESSIONE', re.DOTALL)
    decompress_pattern = re.compile(r'//INIZIO_FUNZIONE_DECOMPRESSIONE\n(.*?)//FINE_FUNZIONE_DECOMPRESSIONE', re.DOTALL)
    
    compress_match = compress_pattern.search(code)        
    decompress_match = decompress_pattern.search(code)        
    
    if not compress_match or not decompress_match:
        raise ValueError("Could not find compression/decompression functions")
    
    return {
        'compress': compress_match.group(1),
        'decompress': decompress_match.group(1)
    }

In [16]:
code = """} CompressionStats;

// Buffer massimo per la lettura/scrittura
#define MAX_BUFFER_SIZE 1048576 // 1 MB

// Funzione di compressione da evolvere
// Implementazione iniziale molto semplice (run-length encoding base)
//INIZIO_FUNZIONE_COMPRESSIONE
unsigned char* compress(const unsigned char* data, size_t data_size, size_t* compressed_size) {
    // Allocazione del buffer di output (nel caso peggiore, la dimensione potrebbe aumentare)
    unsigned char* compressed = (unsigned char*)malloc(data_size * 2);
    if (compressed == NULL) return NULL;
    
    size_t compressed_index = 0;
    size_t i = 0;
    
    // Semplice implementazione RLE
    while (i < data_size) {
        unsigned char current = data[i];
        unsigned char count = 1;
        
        // Conta occorrenze consecutive
        while (i + 1 < data_size && data[i + 1] == current && count < 255) {
            count++;
            i++;
        }
        
        // Memorizza conteggio e valore
        compressed[compressed_index++] = count;
        compressed[compressed_index++] = current;
        
        i++;
    }
    
    // Ridimensiona il buffer alla dimensione effettiva
    compressed = (unsigned char*)realloc(compressed, compressed_index);
    *compressed_size = compressed_index;
    
    return compressed;
}
//FINE_FUNZIONE_COMPRESSIONE


// Funzione di decompressione da evolvere
//INIZIO_FUNZIONE_DECOMPRESSIONE
unsigned char* decompress(const unsigned char* compressed_data, size_t compressed_size, size_t* decompressed_size) {
    // Stimare la dimensione decompresso (sovrastimata)
    // Nella RLE semplice, possiamo stimare che ogni coppia produce fino a 255 byte
    size_t max_size = compressed_size * 255; 
    unsigned char* decompressed = (unsigned char*)malloc(max_size);
    if (decompressed == NULL) return NULL;
    
    size_t decompressed_index = 0;
    size_t i = 0;
    
    // Decompressione RLE base
    while (i < compressed_size) {
        unsigned char count = compressed_data[i++];
        unsigned char value = compressed_data[i++];
        
        // Ripete il valore 'count' volte
        for (unsigned char j = 0; j < count; j++) {
            decompressed[decompressed_index++] = value;
        }
    }
    
    // Ridimensiona alla dimensione effettiva
    decompressed = (unsigned char*)realloc(decompressed, decompressed_index);
    *decompressed_size = decompressed_index;
    
    return decompressed;
}
//FINE_FUNZIONE_DECOMPRESSIONE


// Funzione per leggere dati da file
unsigned char* read_file(const char* filename, size_t* size) {
     """

In [17]:
import re
functions = extract_functions(code = code)
print(functions['compress'])

unsigned char* compress(const unsigned char* data, size_t data_size, size_t* compressed_size) {
    // Allocazione del buffer di output (nel caso peggiore, la dimensione potrebbe aumentare)
    unsigned char* compressed = (unsigned char*)malloc(data_size * 2);
    if (compressed == NULL) return NULL;
    
    size_t compressed_index = 0;
    size_t i = 0;
    
    // Semplice implementazione RLE
    while (i < data_size) {
        unsigned char current = data[i];
        unsigned char count = 1;
        
        // Conta occorrenze consecutive
        while (i + 1 < data_size && data[i + 1] == current && count < 255) {
            count++;
            i++;
        }
        
        // Memorizza conteggio e valore
        compressed[compressed_index++] = count;
        compressed[compressed_index++] = current;
        
        i++;
    }
    
    // Ridimensiona il buffer alla dimensione effettiva
    compressed = (unsigned char*)realloc(compressed, compressed_index);
    *compressed_s

In [3]:
file_text = "example_thinking.txt"
with open(file_text, "r") as file:
    text = file.read()


llm_response = text.split('</think>', 1)[1]
print(llm_response)




//INIZIO_FUNZIONE_COMPRESSIONE
unsigned char* compress(const unsigned char* data, size_t data_size, size_t* compressed_size) {
    unsigned char* compressed = (unsigned char*)malloc(data_size * 2);
    if (compressed == NULL) return NULL;
    
    size_t compressed_index = 0;
    size_t i = 0;
    
    while (i < data_size) {
        if (i + 2 < data_size && data[i] == data[i+1] && data[i] == data[i+2]) {
            unsigned char current = data[i];
            size_t run_length = 3;
            while (i + run_length < data_size && data[i + run_length] == current && run_length < 128) {
                run_length++;
            }
            compressed[compressed_index++] = (run_length - 1) | 0x80;
            compressed[compressed_index++] = current;
            i += run_length;
        } else {
            size_t start = i;
            size_t end = start;
            while (end < data_size && (end - start) < 127) {
                if (end + 2 < data_size && data[end] == data[end+1] 